# Stage 6 — table

Re-create this stage's script with Gemini's help. The cells below give you the spec, the seed, the gotchas, and a verification step. The implementation itself is yours to write.


## 1. Setup

Every cell in this section is idempotent and safe to re-run. If you opened this notebook fresh (without running Stage 0 first in the same runtime), run all of them now.


### 1a. Clone the repo and `cd` into it


In [ ]:
# Bootstrap: clone the workshop repo into /content and cd into it.
# Idempotent — safe to re-run.
import os, subprocess, sys
REPO_DIR = "/content/ar-bic-2026-workshop"
if not os.path.exists(REPO_DIR):
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/jayprimer/ar-bic-2026-workshop.git", REPO_DIR],
        check=True,
    )
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())


### 1b. Install dependencies

Python (`openai`) and the Node CLI `@llamaindex/liteparse`. First run takes ~30s; re-runs are near-instant.


In [ ]:
# Install dependencies. Idempotent (pip skips already-installed; npm re-link is cheap).
# liteparse only matters for Stage 4 but installing it everywhere keeps each
# notebook self-contained, which is the whole point of re-running this cell.
!pip install -q -r requirements.txt
!npm install -g @llamaindex/liteparse 2>&1 | tail -3


### 1c. (no API key needed for this stage)


In [ ]:
# This stage doesn't call the OpenAI API.


### 1d. Stage the bundle-shipped configs


In [ ]:
# Copy bundle-shipped configs into the directories each stage script expects.
# Each stage's input.txt / criteria.txt / schema.json lives under configs/
# in the repo; the actual scripts read them relative to cwd.
import os, shutil
os.makedirs("stage_01", exist_ok=True)
os.makedirs("stage_02", exist_ok=True)
shutil.copy("configs/stage_01_input.txt",    "stage_01/input.txt")
shutil.copy("configs/stage_02_input.txt",    "stage_02/input.txt")
shutil.copy("configs/stage_02_criteria.txt", "stage_02/criteria.txt")
shutil.copy("configs/schema.json",           "schema.json")
print("configs staged")


### 1e. Load prior stages' reference outputs

Stage 6 reads outputs from earlier stages. Each Colab notebook gets its own runtime, so work done in another notebook is not visible here. This cell seeds `stage_01..stage_05/data/` from the canonical reference run so Stage 6 has inputs to work with.


In [ ]:
# Load prior stages' reference outputs as inputs for Stage 6.
# Each Colab notebook opens with a fresh runtime, so any work done in a
# Stage <6 notebook in a DIFFERENT runtime is not visible here.
# This cell makes the stage runnable in isolation against the canonical
# reference run. If you re-run an earlier stage IN THIS runtime, your
# output replaces these reference files (cwd is /content/...).
import os, shutil, glob
for n in range(1, 6):
    dst = f"stage_0{n}/data"
    src = f"reference_outputs/stage_0{n}/data"
    if not os.path.isdir(src):
        continue
    os.makedirs(dst, exist_ok=True)
    # Only seed if the participant hasn't produced anything for this stage
    # in the current runtime — otherwise we'd clobber their work.
    if any(os.scandir(dst)):
        print(f"skip stage_0{n} — already has files (keeping your work)")
        continue
    for src_file in glob.glob(f"{src}/*"):
        shutil.copy(src_file, dst)
    print(f"seeded stage_0{n}/data from reference_outputs")


## 2. Spec — paste this into Gemini

Open the Gemini side panel in Colab (sparkles icon, top right) and paste the block below as your prompt. Then iterate.

```
Flatten every `stage_05/data/*.json` into one row per
(paper × animal_arm) and write
`stage_06/data/mabs_animal_studies.csv` with these columns (in order):

  pmid, source_type, first_author, year, mab_name, target, format,
  development_stage, regulatory_context, threeRs_mentioned,
  author_reduction_recommendation, species, n_animals, study_type,
  duration_days, species_justification, cross_reactivity_evidence,
  endpoints_unique_to_animal, concurrent_nam, n_nams_discussed

`n_nams_discussed` is `len(rec["nams_discussed"])` — a count, not a list.

A paper with no animal_arms contributes zero rows. The header is always
written.

Skip any sibling JSON named `eval.json`, `eval_script.json`,
`eval_llm.json`, or `score.json`.
```


## 3. Gotchas Gemini probably won't know

Copy any that apply into Gemini if it goes off-track:

- **Use `csv.DictWriter(f, fieldnames=FIELDNAMES)`.** Keeps column
  order deterministic even when some arms are missing fields.
- **Open with `newline=""`** to avoid blank lines on Windows runtimes.
- **A paper-level `nams_discussed` list** needs `len(...)` not the
  list itself written to the cell.
- **Skip eval artifacts in the directory** (see spec).


## 4. Seed — a few lines to anchor Gemini in the right direction


In [ ]:
import csv, glob, json, os
STAGE = "stage_06"
DATA = f"{STAGE}/data"
IN_DATA = "stage_05/data"
os.makedirs(DATA, exist_ok=True)
SKIP_NAMES = {"eval.json", "eval_script.json", "eval_llm.json", "score.json"}


## 5. Your implementation

Drive Gemini to fill this in. Iterate until the verification cell below passes.


In [ ]:
# TODO: implement Stage 6 here.
# Read the spec above. Use the seed cell's imports.
# When done, run the verification cell next.


## 6. Verify


In [ ]:
import csv, os
out = "stage_06/data/mabs_animal_studies.csv"
assert os.path.exists(out), "no CSV produced"
with open(out) as f:
    rows = list(csv.DictReader(f))
for r in rows:
    assert r["pmid"], "row missing pmid"
    assert r["species"], "row missing species"
    assert r["study_type"], "row missing study_type"
print(f"OK — {len(rows)} animal-arm rows")


## 7. Run the eval grader

The eval reads only your stage's output and writes `stage_06/eval/eval_*.json` + `score.json`.


In [ ]:
!python eval/eval_06_script.py


## 8. Stuck? Skip this stage

Copy the reference run's Stage 6 output into place so the next stage's notebook can still run. Use this sparingly — the point of the workshop is to *re-create* each stage.


In [ ]:
import os, shutil
os.makedirs("stage_06/data", exist_ok=True)
shutil.copy("reference_outputs/stage_06/data/mabs_animal_studies.csv",
            "stage_06/data/mabs_animal_studies.csv")
print("copied reference Stage 6 output")
